# NB_lab_test_trends

Builds the **gold.lab_test_trends** table — one row per test type per calendar month with abnormal/critical rates and month-over-month trend via LAG. Full rebuild (CREATE OR REPLACE).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresyn.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresyn.gold.lab_test_trends AS

WITH monthly_agg AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date)                                                AS year,
    MONTH(lr.test_date)                                               AS month,
    COUNT(*)                                                          AS total_tests,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)        AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)        AS critical_test_count
  FROM adb_caresyn.silver.lab_results lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),

with_rates AS (
  SELECT
    ma.*,
    ROUND(ma.abnormal_test_count * 100.0 / ma.total_tests, 2)        AS abnormal_rate_pct,
    ROUND(ma.critical_test_count * 100.0 / ma.total_tests, 2)        AS critical_rate_pct,
    LAG(ROUND(ma.abnormal_test_count * 100.0 / ma.total_tests, 2))
      OVER (PARTITION BY ma.test_name ORDER BY ma.year, ma.month)    AS prev_month_abnormal_rate_pct
  FROM monthly_agg ma
)

SELECT
  wr.test_name,
  wr.year,
  wr.month,
  c.month_name,
  CONCAT('Q', c.quarter)                                             AS quarter_name,
  wr.total_tests,
  wr.abnormal_test_count,
  wr.critical_test_count,
  CAST(wr.abnormal_rate_pct           AS DECIMAL(5,2))               AS abnormal_rate_pct,
  CAST(wr.critical_rate_pct           AS DECIMAL(5,2))               AS critical_rate_pct,
  CAST(wr.prev_month_abnormal_rate_pct AS DECIMAL(5,2))              AS prev_month_abnormal_rate_pct,
  CAST(
    CASE
      WHEN wr.prev_month_abnormal_rate_pct IS NULL THEN NULL
      ELSE ROUND(wr.abnormal_rate_pct - wr.prev_month_abnormal_rate_pct, 2)
    END
  AS DECIMAL(6,2))                                                   AS abnormal_rate_change_pct

FROM with_rates wr

LEFT JOIN adb_caresyn.silver.calendar c
  ON c.date_key = CAST(DATE_FORMAT(MAKE_DATE(wr.year, wr.month, 1), 'yyyyMMdd') AS INT);

In [0]:
updated_df = spark.table("adb_caresyn.gold.lab_test_trends")
record_count = updated_df.count()

dbutils.notebook.exit(str(record_count))